# Get Album, Tracks, and Audio Features

We will use Spotify's Web API to get data for the album of choice. The album and artist variables are set in `consts.json`. We will authenticate to Spotify using our `client_id` and `client_secret` set up through our account. Then, get the target album's Spotify ID using Spotify's `search` endpoint. Afterwards, using the album's id, get it's album-level and track-level data.

Spotify recently deprecated their `audio-features` and `audio-analysis` endpoints, much to the community's dismay. Instead, we will use [Reccobeats API](https://reccobeats.com/docs/apis/reccobeats-api) for getting `audio-features` data for each track, which takes the `spotify-track-id` as a query parameter in the request.

In the response we'll get feature metrics like `key`, `tempo`, `energy`, etc. These are Reccobeats [docs for audio-features](https://reccobeats.com/docs/apis/get-audio-features).

For a reminder on how to authenticate to Spotify's API, read their ["Getting Started" docs](https://developer.spotify.com/documentation/web-api/tutorials/getting-started).

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
SPOTIFY_CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')

In [4]:
import requests
import json

In [5]:
# Authenticate to Spotify and request an access token

url = 'https://accounts.spotify.com/api/token'

headers = {
    "Content-Type": "application/x-www-form-urlencoded"
}

payload = {
    "grant_type": "client_credentials",
    "client_id": SPOTIFY_CLIENT_ID,
    "client_secret": SPOTIFY_CLIENT_SECRET
}

res = requests.post(url, headers=headers, data=payload)
res.json()

{'access_token': 'BQBne9ko76kPxWj6ZXj6JypfTJoIC4BShTZi_46ewQKF_sTYhPLjnOASCzWeEqg1X0lv1MJAyUGs0Qgq_7R_DX1XDlwiffioOWv0c1lTJd_kOfVvHTuMpV28JAX1aNG9JozSBnZ65vBi',
 'token_type': 'Bearer',
 'expires_in': 3600}

In [6]:
spotify_access_token = res.json()["access_token"]

In [7]:
with open('consts.json', 'r') as file:
    data = json.load(file)

album = data['album']
artist = data['artist']

In [21]:
url = 'https://api.spotify.com/v1/search'

headers = {
    "Authorization": f'Bearer {spotify_access_token}'
}

query = f'album:"{album}" artist:"{artist}"'

params = {
    'q': query,
    'type': 'album'
}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    if data['albums']['items']:
        spotify_album_id = data['albums']['items'][0]['id']
        print(spotify_album_id)
    else:
        print("No album found from inputs ", album, "&", artist)
else:
    raise Exception(f"API Error: {response.status_code}")

72GMOkq47rXzdAMJrjf4RV


In [22]:
# Get album data
url = f'https://api.spotify.com/v1/albums/{spotify_album_id}'

res = requests.get(url, headers=headers)
res.json()

{'album_type': 'album',
 'total_tracks': 17,
 'external_urls': {'spotify': 'https://open.spotify.com/album/72GMOkq47rXzdAMJrjf4RV'},
 'href': 'https://api.spotify.com/v1/albums/72GMOkq47rXzdAMJrjf4RV',
 'id': '72GMOkq47rXzdAMJrjf4RV',
 'images': [{'url': 'https://i.scdn.co/image/ab67616d0000b2737bf7e051d397970737661d05',
   'height': 640,
   'width': 640},
  {'url': 'https://i.scdn.co/image/ab67616d00001e027bf7e051d397970737661d05',
   'height': 300,
   'width': 300},
  {'url': 'https://i.scdn.co/image/ab67616d000048517bf7e051d397970737661d05',
   'height': 64,
   'width': 64}],
 'name': 'Like Water For Chocolate',
 'release_date': '2000-03-28',
 'release_date_precision': 'day',
 'type': 'album',
 'uri': 'spotify:album:72GMOkq47rXzdAMJrjf4RV',
 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/2GHclqNVjqGuiE5mA7BEoc'},
   'href': 'https://api.spotify.com/v1/artists/2GHclqNVjqGuiE5mA7BEoc',
   'id': '2GHclqNVjqGuiE5mA7BEoc',
   'name': 'Common',
   'type': 'arti

In [8]:
album_data = res.json()

In [9]:
album_imgs = album_data["images"]
album_name = album_data["name"]
album_release_date = album_data["release_date"]
album_artist = album_data["artists"][0]["name"]
album_id = album_data["id"]
album_artist_id = album_data["artists"][0]["id"]
album_artist_href = album_data["artists"][0]["href"]

In [10]:
album_tracks = album_data["tracks"]["items"]

In [11]:
tracks = []
for track in album_tracks:
    spotify_track_id = track["id"]
    track_name = track["name"]
    track_number = track["track_number"]

    track_meta = {
        "spotify_track_id": spotify_track_id,
        "track_name": track_name,
        "track_number": track_number
    }
    tracks.append(track_meta)

In [12]:
# Get track features from Reccobeats API. Spotify's endpoints have been deprecated.

headers = {
  'Accept': 'application/json'
}

tracks_w_audio_feats = []
for tm in tracks:
    url = f"https://api.reccobeats.com/v1/audio-features?ids={tm["spotify_track_id"]}"
    res = requests.get(url, headers=headers)
    audio_feats_meta = res.json()["content"][0]

    track_w_audio_feats = {**tm, **audio_feats_meta}
    tracks_w_audio_feats.append(track_w_audio_feats)

tracks_w_audio_feats

[{'spotify_track_id': '2AaB2ZDeJXu6j4Csos4gZH',
  'track_name': "Time Travelin' (A Tribute To Fela)",
  'track_number': 1,
  'id': '6382aa7b-2fdb-4251-8dc5-18903a522332',
  'href': 'https://open.spotify.com/track/2AaB2ZDeJXu6j4Csos4gZH',
  'isrc': 'USMC10000119',
  'acousticness': 0.227,
  'danceability': 0.83,
  'energy': 0.551,
  'instrumentalness': 0.254,
  'key': 7,
  'liveness': 0.376,
  'loudness': -14.559,
  'mode': 1,
  'speechiness': 0.283,
  'tempo': 104.179,
  'valence': 0.434},
 {'spotify_track_id': '1ZD3CMagZCxFyrc2zPyvBl',
  'track_name': 'Heat',
  'track_number': 2,
  'id': 'd3f7c3de-043e-4242-b630-7ec5af6083f0',
  'href': 'https://open.spotify.com/track/1ZD3CMagZCxFyrc2zPyvBl',
  'isrc': 'USMC10000120',
  'acousticness': 0.0057,
  'danceability': 0.811,
  'energy': 0.732,
  'instrumentalness': 0.0312,
  'key': 11,
  'liveness': 0.0929,
  'loudness': -7.814,
  'mode': 0,
  'speechiness': 0.262,
  'tempo': 102.587,
  'valence': 0.627},
 {'spotify_track_id': '6Is1oWZB3Ry1b

In [13]:
len(tracks_w_audio_feats)

16

In [14]:
import pandas as pd

In [15]:
df = pd.DataFrame(tracks_w_audio_feats)
df

,spotify_track_id,track_name,track_number,id,href,isrc,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence
0,2AaB2ZDeJXu6j4Csos4gZH,Time Travelin' (A Tribute To Fela),1,6382aa7b-2fdb-4251-8dc5-18903a522332,https://open.spotify.com/track/2AaB2ZDeJXu6j4C...,USMC10000119,0.2270,0.830,0.551,0.254000,7,0.3760,-14.559,1,0.283,104.179,0.434
1,1ZD3CMagZCxFyrc2zPyvBl,Heat,2,d3f7c3de-043e-4242-b630-7ec5af6083f0,https://open.spotify.com/track/1ZD3CMagZCxFyrc...,USMC10000120,0.0057,0.811,0.732,0.031200,11,0.0929,-7.814,0,0.262,102.587,0.627
2,6Is1oWZB3Ry1bbMx2MKeui,Cold Blooded,3,cbd193e9-63ec-4fca-b460-48bf0973e87f,https://open.spotify.com/track/6Is1oWZB3Ry1bbM...,USMC10000121,0.3400,0.717,0.903,0.000180,4,0.1090,-6.395,0,0.351,98.829,0.456
3,7bEEzWWgJS4HhzYtNLCXfa,Dooinit,4,aac586dc-d7de-408d-8580-d00a5cc52af6,https://open.spotify.com/track/7bEEzWWgJS4HhzY...,USMC10000122,0.1020,0.850,0.637,0.000000,5,0.0995,-5.820,0,0.393,92.859,0.760
4,5NiUrZVKyLpsyj62Roq5FW,The Light,5,21901fdd-9963-4813-89d4-29238a40df59,https://open.spotify.com/track/5NiUrZVKyLpsyj6...,USMC10000123,0.0343,0.939,0.727,0.000000,4,0.0855,-4.349,0,0.235,96.968,0.660
5,3aoSjb0bYD2pR2h7R4UzAj,Funky For You,6,f6072dab-961b-468d-86ed-9b3b3bae6b31,https://open.spotify.com/track/3aoSjb0bYD2pR2h...,USMC10000124,0.1380,0.542,0.706,0.000020,11,0.1660,-6.678,1,0.542,98.674,0.689
6,12DQLP0EURlKcwguEJM5oY,The Questions,7,07a320f6-a73c-4ab6-97b7-095828d94163,https://open.spotify.com/track/12DQLP0EURlKcwg...,USMC10000125,0.3700,0.828,0.388,0.000000,7,0.3960,-12.355,1,0.333,90.742,0.820
7,3cwWEfm58FQbdJBM4BLI43,Time Travelin Reprise,8,87fd629f-4015-4c9d-ab2c-afd4554c2426,https://open.spotify.com/track/3cwWEfm58FQbdJB...,USMC10000126,0.0168,0.904,0.332,0.819000,7,0.3200,-15.613,1,0.194,104.232,0.690
8,6OE9S6XF0U1lNfeaUNSjYl,The 6th Sense,9,421b4d21-7b24-4ca9-b695-13e8dca916af,https://open.spotify.com/track/6OE9S6XF0U1lNfe...,USMC10000134,0.0746,0.687,0.827,0.000000,1,0.7020,-6.301,1,0.377,94.441,0.882
9,71gd3cOuLvtmUb17OARvsJ,A Film Called (Pimp),10,f277a79e-6496-4433-9f02-2c74d11e68db,https://open.spotify.com/track/71gd3cOuLvtmUb1...,USMC10000127,0.0915,0.788,0.664,0.000005,10,0.5080,-5.047,0,0.372,78.294,0.763


In [23]:
script_dir = os.path.dirname(os.path.abspath('get-album-tracks-audio-features.ipynb'))
output_file_path = os.path.join(script_dir, '..', 'data', f'{album}', 'output', f'track_list_w_audio_feats_{album}.csv')

In [16]:
df.to_csv(output_file_path, index=False)